In [1]:
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json


from pathlib import Path

# Preliminari

In [2]:
# Configurazione OpenAI
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_4
TEMPERATURE = 0.0

MODEL = constants.OPENAI_GPT_5_NANO
RESULTS_FILE_NAME = 'results_' + 'gpt-5-nano' + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [3]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.

Il tuo compito è estrarre informazioni strutturate dal referto RM fornito e restituire esclusivamente un oggetto JSON valido conforme allo schema seguente:

{"morfologia": "solido_polipoide | solido_anulare | mucinoso", "ore_inizio": "int | null", "ore_fine": "int | null", "spessore_parietale": "int | null", "estensione_cranio_caudale": "int | null", "distanza_oai": "int | null", "posizione": {"basso": "no | si", "medio": "no | si", "alto": "no | si", "giunzione": "no | si"}, "riflessione_peritoneale_anteriore": "sotto | cavallo | non_valutabile", "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus", "infiltrazione_sfinteri": "no | si", "infiltrazione_organi_extra": "no | si", "infiltrazione_organi_dettagli": {"pavimento_pelvico": "no | si", "altro": "no | si"}, "coinvolgimento_riflessione_peritoneale": "no | si", "coinvolgimento_fascia_mesorettale": "no | si", "numero_linfonodi_no

# Load Data

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(test_data) = 65
len(validation_data) = 66


In [5]:
print(validation_data.iloc[0].report_text)

A LIVELLO DEL RETTO MEDIO-ALTO, A CIRCA 10 CM DAL MARGINE ANALE ESTERNO, SI DOCUMENTA GROSSOLANO ISPESSIMENTO PARIETALE CHE INTERESSA CIRCONFERENZIALMENTE IL VISCERE IN SEDE CENTRALE (CON ASPETTO STENOTICO) ED I 3/4 DELLA CIRCONFERENZA IN SEDE MARGINALE SUPERIORE ED INFERIORE, CON SPESSORE MASSIMO DI 27 MM ED ESTENSIONE LONGITUDINALE DI CIRCA 65 MM. L'ISPESSIMENTO E CARATTERIZZATO DA SEGNALE IPOINTENSO NELLE IMMAGINI T2-DIPENDENTI E MARCATA RESTRIZIONE DELLA DIFFUSIONE, CON SIGNIFICATO PATOLOGICO, E MOSTRA ESTESI SEGNI DI ESTENSIONE EXTRA-PARIETALE NEL SUO III MEDIO LUNGO IL VERSANTE ANTERIO-LATERALE SINISTRO (CON GROSSOLANO GETTONE SOLIDO DI 24 MM) OVE IL TESSUTO PATOLOGICO RAGGIUNGE ED INFILTRA LA FASCIA MESORETTALE E LA RIFLESSIONE PERITONEALE. A TALE LIVELLO IL TESSUTO GIUNGE IN ADIACENZA AL PROFILO POSTERIORE DEL CORPO UTERINO, CON APPARENTE CONSERVATO PIANO DI CLIVAGGIO ADIPOSO. 
MULTIPLI LINFONODI (> 6) A MORFOLOGIA ROTONDEGGIANTE ED ASSE CORTO MASSIMO DI 6 MM IN SEDE MESORETTAL

# Generazione

## Preliminari generazione

In [13]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             report_text: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = PYDANTIC_MODEL):
    response = client.responses.parse(
        model=model,
        top_p=0,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
            
        ],
        text_format=output_format
    )
    return response

In [10]:
help(client.responses.parse)

Help on method parse in module openai.resources.responses.responses:

parse(*, text_format: 'type[TextFormatT] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, background: 'Optional[bool] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, conversation: 'Optional[response_create_params.Conversation] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, include: 'Optional[List[ResponseIncludable]] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, input: 'Union[str, ResponseInputParam] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, instructions: 'Optional[str] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, max_output_tokens: 'Optional[int] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, max_tool_calls: 'Optional[int] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, metadata: 'Optional[Metadata] | Omit' = <openai.Omit object at 0x000001881F4D29D0>, model: 'ResponsesModel | Omit' = <openai.Omit object at 0x000001881F4D29D0>, parallel_tool_calls: 'Optiona

In [11]:
# Esempio
if True:
    result = extract_data_from_report(MODEL, data['test'].iloc[0]['report_text'])

In [12]:
if True: # To see the full output
    pprint(result.model_dump())
if False:  # To see the string output text
    print(result.output_text)
if False:  # To see the parsed output as a pydantic BaseModel
    print(result.output_parsed)
if False:
    print(result.output_parsed.model_dump(mode='json'))

{'background': False,
 'billing': {'payer': 'developer'},
 'completed_at': 1771006268,
 'conversation': None,
 'created_at': 1771006171.0,
 'error': None,
 'frequency_penalty': 0.0,
 'id': 'resp_0abd3a99e74a4fd500698f68dafbc081979796cb66128bddb5',
 'incomplete_details': None,
 'instructions': None,
 'max_output_tokens': None,
 'max_tool_calls': None,
 'metadata': {},
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'response',
 'output': [{'content': None,
             'encrypted_content': None,
             'id': 'rs_0abd3a99e74a4fd500698f68db88fc819798b868ccd5e5a9f0',
             'status': None,
             'summary': [],
             'type': 'reasoning'},
            {'content': [{'annotations': [],
                          'logprobs': [],
                          'parsed': {'coinvolgimento_fascia_mesorettale': 'si',
                                     'coinvolgimento_riflessione_peritoneale': 'no',
                                     'depositi_tumorali': 'si',
                  

## Inference
Usiamo modelli non finetunati. Solo prompt engineering.

In [14]:
print(MODEL)
df = pd.concat([validation_data, test_data], ignore_index=True)
ids = []
actual = []
predicted = []
splits = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits.append(row['split'])
    output = extract_data_from_report(model=MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids.append(row[constants.ID_COMULM_NAME])
    actual.append(real.model_dump(mode='json'))
    if output is None:
        predicted.append('no output')
    else:
        predicted.append(output.output_parsed.model_dump(mode='json'))

gpt-5-nano-2025-08-07


100%|██████████| 131/131 [2:24:54<00:00, 66.37s/it] 


In [15]:
results_dicts = []
assert len(ids) == len(actual) == len(predicted)
for id, act, pred, split in zip(ids, actual, predicted, splits):
    results_dicts.append(
        {
            'model': MODEL,
            'split': split,
            'id': int(id),
            'actual': act,
            'prediction': pred
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'inference' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')

In [16]:
output.model_dump()

{'id': 'resp_0f3b226f2a7edc0000698f8ca477e08194b8e560ef2c87ae73',
 'created_at': 1771015332.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'response',
 'output': [{'id': 'rs_0f3b226f2a7edc0000698f8ca4e9748194b8c5bf1e70044cce',
   'summary': [],
   'type': 'reasoning',
   'content': None,
   'encrypted_content': None,
   'status': None},
  {'id': 'msg_0f3b226f2a7edc0000698f8cd8b4cc8194aae667cf6d53f831',
   'content': [{'annotations': [],
     'text': '{"morfologia":"solido_anulare","ore_inizio":null,"ore_fine":null,"spessore_parietale":null,"estensione_cranio_caudale":40,"distanza_oai":50,"posizione":{"basso":"no","medio":"si","alto":"no","giunzione":"no"},"riflessione_peritoneale_anteriore":"non_valutabile","infiltrazione_tessuto_adiposo":"si_5mm","infiltrazione_sfinteri":"no","infiltrazione_organi_extra":"no","infiltrazione_organi_dettagli":{"pavimento_pelvico":"no","altro":"no"},"coinvolgimento_rif